In [6]:
#Classifier data load
import pandas as pd
import pickle

exp_data = pd.read_csv(r"/home/lero/idrive/cmac/DDMAP/Stability studies/Stability_dataset_August_update.csv", na_values='nan')

with open('/home/lero/idrive/cmac/DDMAP/Stability studies/Model_results/Classifier/July_api_no/Classifiers_results_dictionary.pkl', 'rb') as f:
    results = pickle.load(f)
model_data = pd.DataFrame(results['XGBoost classifier']['predictions'], columns=['RF_stability'])

non_binary = exp_data['Average days stable']
binary = (non_binary>=2160).astype(int)
exp_data['binary_stability'] = binary
exp_data['predictions'] = model_data

#drop pure systems as we are estimating best polymers
exp_data = exp_data[exp_data['Polymer']!='Pure']

#exp_data

In [7]:
#calculate stable systems for truth data
truth_most_stable_polymers = {}

for plate, group in exp_data.groupby(['API', 'condition']):
    stable = group[group['binary_stability']==1]
    max_stable_loading = stable.groupby('Polymer')['Drug loading (wt%)'].max()
    if max_stable_loading.empty:
        truth_most_stable_polymers[plate]='nan'
        continue

    highest_loading = max_stable_loading.max()
    best_polymers = set(max_stable_loading[max_stable_loading == highest_loading].index)
    truth_most_stable_polymers[plate] = best_polymers

#truth_most_stable_polymers



In [8]:
#calculate stable systems for experimental data

pred_most_stable_polymers={}

for plate, group in exp_data.groupby(['API', 'condition']):
    stable = group[group['predictions']==1]
    max_stable_loading = stable.groupby('Polymer')['Drug loading (wt%)'].max()
    if max_stable_loading.empty:
        pred_most_stable_polymers[plate]='nan'
        continue
    
    highest_loading = max_stable_loading.max()
    best_polymers = set(max_stable_loading[max_stable_loading == highest_loading].index)
    pred_most_stable_polymers[plate] = best_polymers
    
pred_most_stable_polymers


{('AZ', '30_C_30_RH'): {'Affinisol',
  'HPMCAS',
  'PVP_K30',
  'Plasdone',
  'Soluplus'},
 ('AZ', '40_C_0_RH'): {'Affinisol',
  'HPMCAS',
  'PVP_K30',
  'Plasdone',
  'Soluplus'},
 ('AZ', '40_C_75_RH'): {'Affinisol',
  'HPMCAS',
  'PVP_K30',
  'Plasdone',
  'Soluplus'},
 ('Acetaminophen', '30_C_30_RH'): {'PVP_K30', 'Plasdone', 'Soluplus'},
 ('Acetaminophen', '40_C_0_RH'): {'Affinisol',
  'HPMCAS',
  'PVP_K30',
  'Plasdone',
  'Soluplus'},
 ('Acetaminophen', '40_C_75_RH'): {'Affinisol',
  'HPMCAS',
  'PVP_K30',
  'Plasdone',
  'Soluplus'},
 ('Bifonazole', '30_C_30_RH'): {'Affinisol',
  'PVP_K30',
  'Plasdone',
  'Soluplus'},
 ('Bifonazole', '40_C_0_RH'): {'HPMCAS', 'Plasdone'},
 ('Bifonazole', '40_C_75_RH'): {'PVP_K30', 'Soluplus'},
 ('Carbamazepine', '30_C_30_RH'): {'PVP_K30', 'Plasdone'},
 ('Carbamazepine', '40_C_0_RH'): {'Plasdone'},
 ('Carbamazepine', '40_C_75_RH'): {'Affinisol',
  'HPMCAS',
  'PVP_K30',
  'Soluplus'},
 ('Celecoxib', '30_C_30_RH'): {'Affinisol',
  'HPMCAS',
  'PVP_

In [9]:
#calculate recall @K (K is each plate)

import numpy as np
results = []

for plate in truth_most_stable_polymers:
    true_set = truth_most_stable_polymers[plate]
    pred_set = pred_most_stable_polymers[plate]
    if not true_set or true_set == 'nan':
        continue  # skip if there's no true answer (undefined recall)
    if pred_set == 'nan':
        pred_set = []  # treat nan as empty set
    # Count how many of the true polymers are in the predicted list
    correct = sum(1 for item in true_set if item in pred_set)
    recall = correct/len(true_set)
    #results.append(recall)
    if pred_set:
        precision = correct/len(pred_set)
    else:
        precision = 0
    #results.append(precision)
    results.append({
        'Plate': plate, 
        'Recall': recall,
        'Precision': precision,
        'True_set': true_set,
        'Pred_set': pred_set
    })

results_df = pd.DataFrame(results)
recall_values = results_df['Recall']
Precision_values = results_df['Precision']
recall_values_mean = np.mean(recall_values)
precision_values_mean = np.mean(Precision_values)
print(f'Averaged mean recall: {recall_values_mean:.2f}\nAveraged mean precision: {precision_values_mean:.2f}')

results_df

Averaged mean recall: 0.86
Averaged mean precision: 0.73


,Plate,Recall,Precision,True_set,Pred_set
0,"(AZ, 30_C_30_RH)",1.0,1.000000,"{Plasdone, Affinisol, Soluplus, HPMCAS, PVP_K30}","{Plasdone, Affinisol, Soluplus, HPMCAS, PVP_K30}"
1,"(AZ, 40_C_0_RH)",1.0,1.000000,"{Plasdone, Affinisol, Soluplus, HPMCAS, PVP_K30}","{Plasdone, Affinisol, Soluplus, HPMCAS, PVP_K30}"
2,"(AZ, 40_C_75_RH)",1.0,1.000000,"{Plasdone, Affinisol, Soluplus, HPMCAS, PVP_K30}","{Plasdone, Affinisol, Soluplus, HPMCAS, PVP_K30}"
3,"(Acetaminophen, 30_C_30_RH)",1.0,0.333333,{PVP_K30},"{PVP_K30, Plasdone, Soluplus}"
4,"(Acetaminophen, 40_C_0_RH)",1.0,0.200000,{Plasdone},"{Plasdone, Affinisol, Soluplus, HPMCAS, PVP_K30}"
...,...,...,...,...,...
88,"(Sulfamerazine, 40_C_0_RH)",1.0,0.333333,{Plasdone},"{Soluplus, Plasdone, Affinisol}"
89,"(Sulfamerazine, 40_C_75_RH)",0.2,1.000000,"{Plasdone, Affinisol, Soluplus, HPMCAS, PVP_K30}",{Plasdone}
90,"(Tinidazole, 30_C_30_RH)",1.0,0.200000,{Soluplus},"{Plasdone, Affinisol, Soluplus, HPMCAS, PVP_K30}"
91,"(Tinidazole, 40_C_0_RH)",1.0,0.500000,"{Plasdone, Soluplus}","{HPMCAS, Soluplus, Plasdone, Affinisol}"
